# 🏗️ Inria Building Segmentation — Colab Training 

## ⚠️ Read before running anything

**Set your runtime to GPU first, before running any cell.**
Changing the runtime mid-session wipes `/content` entirely — you would lose downloaded tiles and patches.

```
Runtime → Change runtime type → T4 GPU → Save
```

### Session storage layout

| Path | What | Persists? |
|---|---|---|
| `/content/inria_raw/` | Raw 15GB tiles | ❌ wiped on session end |
| `/content/inria_patches/` | 512×512 patches | ❌ wiped on session end |
| `/content/drive/MyDrive/inria-segmentation/` | Checkpoints + MLflow logs | ✅ Drive |

Only checkpoints and MLflow logs go to Drive (~100MB total). Everything else lives on the VM's local disk (~70GB available) and gets wiped automatically — this does **not** count against your Drive quota.

### Each session flow
1. Set runtime to GPU ← **do this first**
2. Run all cells top to bottom
3. Cells 1–4: setup + GPU check
4. Cell 5: clone repo + install deps
5. Cell 6: mount Drive (checkpoints only)
6. Cell 7: download raw tiles → `/content/inria_raw/`
7. Cell 8: patch tiles → `/content/inria_patches/`
8. Cell 9: verify dataset
9. Cell 10: configure paths (Drive for output only)
10. Cell 11: train → checkpoint auto-saved to Drive
11. Cell 12: evaluate

## 1. GPU check — run this first

In [ ]:
import subprocess 

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)

if result.returncode != 0:
    raise RuntimeError(
        '\n'
        '❌  No GPU detected.\n'
        '    Go to Runtime → Change runtime type → T4 GPU → Save\n'
        '    then re-run from the top.\n'
        '    Do NOT download data first — changing runtime wipes /content.'
    )

print('✅  GPU detected:')
print(result.stdout)

## 2. Check available disk space

In [ ]:
import shutil

total, used, free = shutil.disk_usage('/content')
gb = 1024 ** 3

print(f'VM local disk  →  total: {total/gb:.0f}GB  used: {used/gb:.1f}GB  free: {free/gb:.1f}GB')
print('(This is NOT your Google Drive quota)')

if free / gb < 25:
    print(f'\n⚠️  Less than 25GB free. Raw tiles (~15GB) + patches (~8GB) need ~23GB.')
    print('   Consider using the city-subset option in cell 7.')
else:
    print(f'\n✅  Enough space for full dataset.')

## 3. Check PyTorch + CUDA

In [ ]:
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB' if torch.cuda.is_available() else '')

## 4. Clone repo + install dependencies

In [ ]:
import os

REPO_DIR = '/content/inria-building-segmentation'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/YOUR_USERNAME/inria-building-segmentation {REPO_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt
print('\n✅  Dependencies installed.')

## 5. Mount Google Drive

Drive is used **only** for checkpoints and MLflow logs (~100MB total).
Raw tiles and patches stay on the VM's local disk — they do not count against your Drive quota.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/inria-segmentation'
os.makedirs(f'{DRIVE_OUT}/checkpoints', exist_ok=True)


print(f'\nDrive output directory: {DRIVE_OUT}')
print('  checkpoints/ → model weights (~50MB per checkpoint)')
print('  mlruns/      → MLflow experiment logs (~a few MB)')

total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'\nDrive quota  →  used: {used/gb:.1f}GB  free: {free/gb:.1f}GB')

## 6. Download raw Inria tiles → `/content/inria_raw/`

These go to the VM's local disk. **Not Drive. Not counted against your quota.**

The Inria dataset is downloaded via an official shell script that handles
authentication and splits the archive into parts automatically.
Register once (free) at:
https://project.inria.fr/aerialimagelabeling/download/

The script downloads everything to the **current working directory**,
so we `cd` into `RAW_DIR` first.

> The script pulls ~15GB in several parts and takes 10–20 min on Colab's connection.

In [ ]:
import os

RAW_DIR = '/content/inria_raw'
os.makedirs(RAW_DIR, exist_ok=True)

print(f'Downloading Inria dataset to {RAW_DIR}  (VM local disk — not Drive)...')
print('This takes 10–20 min. The script downloads several archive parts automatically.\n')

# cd into RAW_DIR so the script drops files there, not in /content/repo
%cd {RAW_DIR}
!curl -k https://files.inria.fr/aerialimagelabeling/getAerial.sh | bash

# Return to repo
%cd {REPO_DIR}

print('\nDisk after download:')
!df -h /content

# The script extracts to AerialImageDataset/ inside RAW_DIR
# Detect the actual extracted path
import glob
candidates = glob.glob(f'{RAW_DIR}/**/train/images', recursive=True)
if not candidates:
    raise RuntimeError(
        f'Could not find train/images/ under {RAW_DIR}.\n'
        'Check the output above for errors from the download script.'
    )

TRAIN_SRC = str(candidates[0]).replace('/train/images', '')
print(f'\n✅  Dataset found at: {TRAIN_SRC}')
print(f'Tiles:')
!ls {TRAIN_SRC}/train/images/ | head -20

## 7. Patch tiles → `/content/inria_patches/`

Slices 5000×5000 tiles into 512×512 crops.
Output also stays on VM local disk.

In [ ]:
PATCHES_DIR = '/content/inria_patches'

print('Patching tiles → /content/inria_patches/  (VM local disk)...')

!python scripts/patch_dataset.py \
    --src {TRAIN_SRC}/train \
    --dst {PATCHES_DIR} \
    --size 512 \
    --stride 256

print(f'\nDisk after patching:')
!df -h /content

print(f'\nPatches by city:')
!for city in {PATCHES_DIR}/*/; do echo "  $(basename $city): $(ls $city/images/ | wc -l) patches"; done

## 8. Verify dataset loads correctly

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from data.dataset import InriaDataset
from data.transforms import get_train_transforms

cities = InriaDataset.available_cities(PATCHES_DIR)
print(f'Cities found: {cities}')

ds = InriaDataset(PATCHES_DIR, cities=cities[:1], transform=get_train_transforms())
img, mask = ds[0]
print(f'Sample  →  image: {tuple(img.shape)}  mask: {tuple(mask.shape)}')
print(f'Building pixels: {mask.sum():.0f} / {mask.numel()} ({100*mask.mean():.1f}%)')
print('\n✅  Dataset verified.')

## 9. Configure paths

- Patches  → `/content/inria_patches/`  (VM local disk)
- Checkpoints + MLflow → `/content/drive/MyDrive/inria-segmentation/`  (Drive)

In [ ]:
import yaml

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

# VM local disk — not Drive
cfg['data']['patches_dir'] = PATCHES_DIR

# Drive — only these ~100MB hit your quota
cfg['training']['checkpoint_dir'] = f'{DRIVE_OUT}/checkpoints'
cfg['mlflow']['tracking_uri']     = f'sqlite:///{DRIVE_OUT}/mlruns.db'

# Colab-friendly settings
cfg['training']['num_workers'] = 2

with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Config updated:')
print(f'  patches_dir  → {PATCHES_DIR}  (VM disk)')
print(f'  checkpoints  → {DRIVE_OUT}/checkpoints  (Drive)')
print(f'  mlruns.db    → {DRIVE_OUT}/mlruns.db  (Drive, SQLite)')

## 10. Train

Best checkpoint is saved to Drive automatically whenever val IoU improves.
If this session times out, re-run from cell 1. If a checkpoint exists in Drive, training will resume from this checkpoint, else, will restart from scratch.

In [ ]:
import os

# Use last_checkpoint.pth for resuming (saved every epoch)
# best_model.pth is only for inference (saved on IoU improvement)
LAST_CKPT = f'{DRIVE_OUT}/checkpoints/last_checkpoint.pth'

if os.path.exists(LAST_CKPT):
    print(f'✅  Last checkpoint found on Drive — resuming from epoch where we left off.')
    !python train.py --config configs/config.yaml --resume {LAST_CKPT}
else:
    print('No checkpoint found — starting fresh run.')
    !python train.py --config configs/config.yaml

## 11. Evaluate

In [ ]:
CKPT = f'{DRIVE_OUT}/checkpoints/best_model.pth'
EVAL_OUT = f'{DRIVE_OUT}/evaluation'

!python evaluate.py \
    --checkpoint {CKPT} \
    --config configs/config.yaml \
    --out-dir {EVAL_OUT}

## 12. Show qualitative results

In [ ]:
from IPython.display import Image as IPyImage
IPyImage(filename=f'{DRIVE_OUT}/evaluation/predictions_grid.png')